# WBC CNN Classifier

End-to-end notebook for white blood cell classification using EfficientNet-B0.

**Classes:** EOSINOPHIL · LYMPHOCYTE · MONOCYTE · NEUTROPHIL

**Sections:**
0. Setup
1. Exploratory Data Analysis
2. Data Loading Preview
3. Model Summary
4. Training
5. Evaluation
6. Grad-CAM Visualizations

## 0. Setup

In [ ]:
import sys
from pathlib import Path

# Add src/ to path so we can import our modules directly
sys.path.insert(0, str(Path('..') / 'src'))

%matplotlib inline
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 100

import random
import numpy as np
import torch
from PIL import Image

DATA_DIR = Path('..') / 'data'
RESULTS_DIR = Path('..') / 'results'
RESULTS_DIR.mkdir(exist_ok=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
print(f'PyTorch: {torch.__version__}')

## 1. Exploratory Data Analysis

In [ ]:
from dataset import find_data_root, CLASSES

root = find_data_root(DATA_DIR)
print(f'Data root: {root}')

# Count images per class
counts = {}
for split in ['TRAIN', 'TEST']:
    counts[split] = {}
    for cls in CLASSES:
        cls_dir = root / split / cls
        if cls_dir.is_dir():
            n = len(list(cls_dir.glob('*.jp*g')))
            counts[split][cls] = n

print('\nImage counts per class:')
print(f'{"Class":<15} {"TRAIN":>8} {"TEST":>8}')
print('-' * 33)
for cls in CLASSES:
    tr = counts.get('TRAIN', {}).get(cls, 0)
    te = counts.get('TEST', {}).get(cls, 0)
    print(f'{cls:<15} {tr:>8} {te:>8}')

In [ ]:
import seaborn as sns

# Class distribution bar chart
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, split in zip(axes, ['TRAIN', 'TEST']):
    split_counts = counts.get(split, {})
    if split_counts:
        sns.barplot(x=list(split_counts.keys()), y=list(split_counts.values()), ax=ax, palette='Blues_d')
        ax.set_title(f'{split} class distribution')
        ax.set_ylabel('Image count')
        ax.set_xlabel('Class')
        for bar, v in zip(ax.patches, split_counts.values()):
            ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 20, str(v),
                    ha='center', va='bottom', fontsize=10)

fig.tight_layout()
plt.show()

In [ ]:
# 4x4 grid: one row per class, 4 random samples
n_samples = 4
fig, axes = plt.subplots(len(CLASSES), n_samples, figsize=(3 * n_samples, 3 * len(CLASSES)))

for row, cls in enumerate(CLASSES):
    cls_dir = root / 'TRAIN' / cls
    all_imgs = list(cls_dir.glob('*.jp*g'))
    selected = random.sample(all_imgs, min(n_samples, len(all_imgs)))
    for col, img_path in enumerate(selected):
        ax = axes[row][col]
        ax.imshow(Image.open(img_path).convert('RGB'))
        ax.axis('off')
        if col == 0:
            ax.set_ylabel(cls, fontsize=12, rotation=0, labelpad=60, va='center')

fig.suptitle('Sample Images per Class (TRAIN)', fontsize=14, y=1.01)
fig.tight_layout()
plt.show()

In [ ]:
# Check image size consistency
print('Image size check (10 random images per class):')
for cls in CLASSES:
    cls_dir = root / 'TRAIN' / cls
    sample_imgs = random.sample(list(cls_dir.glob('*.jp*g')), min(10, len(list(cls_dir.glob('*.jp*g')))))
    sizes = [Image.open(p).size for p in sample_imgs]  # (W, H)
    widths, heights = zip(*sizes)
    print(f'  {cls:<15} W: {min(widths)}-{max(widths)}, H: {min(heights)}-{max(heights)}')

## 2. Data Loading Preview

In [ ]:
from dataset import build_dataloaders, IMAGENET_MEAN, IMAGENET_STD
import torchvision

train_loader, val_loader, test_loader = build_dataloaders(
    DATA_DIR, batch_size=8, num_workers=2
)
print(f'Train batches: {len(train_loader)}')
print(f'Val   batches: {len(val_loader)}')
print(f'Test  batches: {len(test_loader)}')

In [ ]:
# Display one batch of augmented training images (de-normalized)
images, labels = next(iter(train_loader))

mean = torch.tensor(IMAGENET_MEAN).view(3, 1, 1)
std  = torch.tensor(IMAGENET_STD).view(3, 1, 1)
denorm = (images * std + mean).clamp(0, 1)

grid = torchvision.utils.make_grid(denorm, nrow=4, padding=4)
plt.figure(figsize=(12, 4))
plt.imshow(grid.permute(1, 2, 0).numpy())
plt.axis('off')
plt.title('Augmented training batch')
plt.show()

## 3. Model Summary

In [ ]:
from model import create_model, count_parameters, freeze_backbone

model = create_model(num_classes=4, pretrained=True)
params = count_parameters(model)
print(f"Total parameters:     {params['total']:,}")
print(f"Trainable parameters: {params['trainable']:,}")

# Show what changes after freezing the backbone
freeze_backbone(model)
frozen_params = count_parameters(model)
print(f"\nAfter freeze_backbone():")
print(f"Trainable parameters: {frozen_params['trainable']:,} (head only)")

## 4. Training

This runs the full training loop inline (same logic as `src/train.py`).

In [ ]:
from train import (
    set_seed, get_optimizer, train_one_epoch, validate,
    save_checkpoint, plot_training_curves
)
from model import create_model, freeze_backbone, unfreeze_backbone, count_parameters

# Hyperparameters — adjust as needed
EPOCHS       = 20
WARMUP       = 5
BATCH_SIZE   = 32
LR           = 1e-3
WEIGHT_DECAY = 1e-4
SEED         = 42

set_seed(SEED)

train_loader, val_loader, test_loader = build_dataloaders(
    DATA_DIR, batch_size=BATCH_SIZE, num_workers=2, seed=SEED
)

model = create_model(num_classes=4, pretrained=True).to(device)
criterion = torch.nn.CrossEntropyLoss()
scaler = torch.cuda.amp.GradScaler(enabled=device.type == 'cuda')

best_val_acc = 0.0
history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}

# Phase 1: Warm-up
print('=== Warm-up phase ===')
freeze_backbone(model)
optimizer = get_optimizer(model, LR, WEIGHT_DECAY, phase='warmup')

for epoch in range(WARMUP):
    train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device, scaler)
    val_loss, val_acc = validate(model, val_loader, criterion, device)
    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['train_acc'].append(train_acc)
    history['val_acc'].append(val_acc)
    print(f'Epoch {epoch+1}/{EPOCHS}  train={train_acc:.4f}  val={val_acc:.4f}')
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        save_checkpoint({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scheduler_state_dict': None,
            'best_val_acc': best_val_acc,
            'class_to_idx': train_loader.dataset.subset.dataset.class_to_idx,
            'args': {},
        }, RESULTS_DIR / 'best_model.pth')
        print(f'  --> Saved (val acc {best_val_acc:.4f})')

# Phase 2: Fine-tune
print('\n=== Fine-tune phase ===')
unfreeze_backbone(model)
optimizer = get_optimizer(model, LR, WEIGHT_DECAY, phase='finetune')
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS - WARMUP)

for epoch in range(WARMUP, EPOCHS):
    train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device, scaler)
    val_loss, val_acc = validate(model, val_loader, criterion, device)
    scheduler.step()
    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['train_acc'].append(train_acc)
    history['val_acc'].append(val_acc)
    print(f'Epoch {epoch+1}/{EPOCHS}  train={train_acc:.4f}  val={val_acc:.4f}  lr={scheduler.get_last_lr()[0]:.2e}')
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        save_checkpoint({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scheduler_state_dict': scheduler.state_dict(),
            'best_val_acc': best_val_acc,
            'class_to_idx': train_loader.dataset.subset.dataset.class_to_idx,
            'args': {},
        }, RESULTS_DIR / 'best_model.pth')
        print(f'  --> Saved (val acc {best_val_acc:.4f})')

print(f'\nBest val acc: {best_val_acc:.4f}')

In [ ]:
# Plot training curves inline
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
epochs_range = range(1, EPOCHS + 1)

ax1.plot(epochs_range, history['train_loss'], label='Train')
ax1.plot(epochs_range, history['val_loss'], label='Val')
ax1.axvline(x=WARMUP + 0.5, color='gray', linestyle='--', label='Warmup end')
ax1.set_title('Loss'); ax1.set_xlabel('Epoch'); ax1.legend()

ax2.plot(epochs_range, history['train_acc'], label='Train')
ax2.plot(epochs_range, history['val_acc'], label='Val')
ax2.axvline(x=WARMUP + 0.5, color='gray', linestyle='--', label='Warmup end')
ax2.set_title('Accuracy'); ax2.set_xlabel('Epoch'); ax2.legend()

fig.tight_layout()
plt.show()

## 5. Evaluation

In [ ]:
from evaluate import load_model_from_checkpoint, run_inference, plot_confusion_matrix
from sklearn.metrics import classification_report

model_eval, ckpt = load_model_from_checkpoint(RESULTS_DIR / 'best_model.pth', device)
class_to_idx = ckpt['class_to_idx']
idx_to_class = {v: k for k, v in class_to_idx.items()}
class_names = [idx_to_class[i] for i in range(len(idx_to_class))]

print('Running inference on test set...')
preds, labels, probs = run_inference(model_eval, test_loader, device)

print('\n=== Classification Report ===')
print(classification_report(labels, preds, target_names=class_names, digits=4))

In [ ]:
# Inline confusion matrix (output_path=None → plt.show())
# First switch to inline backend
import matplotlib
matplotlib.use('module://matplotlib_inline.backend_inline')

import importlib, evaluate as ev_module
importlib.reload(ev_module)  # reload to pick up inline backend

from evaluate import plot_confusion_matrix
plot_confusion_matrix(labels, preds, class_names, output_path=None)

## 6. Grad-CAM Visualizations

In [ ]:
from evaluate import generate_gradcam
from dataset import find_data_root, get_transforms
from torchvision import datasets

root = find_data_root(DATA_DIR)
test_dataset = datasets.ImageFolder(
    root=str(root / 'TEST'), transform=get_transforms('test')
)

generate_gradcam(
    model=model_eval,
    test_dataset=test_dataset,
    class_to_idx=class_to_idx,
    output_dir=RESULTS_DIR,
    n_samples_per_class=4,
    device=device,
)
print('Done.')

In [ ]:
# Display saved Grad-CAM images
from IPython.display import Image as IPImage, display

for cls in CLASSES:
    cam_path = RESULTS_DIR / f'gradcam_{cls.lower()}.png'
    if cam_path.exists():
        print(f'--- {cls} ---')
        display(IPImage(filename=str(cam_path), width=500))